In [11]:
import nibabel as nib        # For nifti files
import numpy as np           # For matrix math
import SimpleITK as sitk     # For N4 correction
import torchio as tio        # For deep learning
from dcm2niix import main as dcm2niix_run
from pathlib import Path
from dotenv import load_dotenv
import os
import ants
from nilearn.datasets import MNI152_FILE_PATH
import torch

#pipeline:
# 1. motion correction and n4 bias field correction

# 2. Skull stripping

# 3. spatial normalization

# 4. Intensity normalization

# 5. resizing (use placeholder values to signify dims)

# 6. Gaussian filters to smooth and denois data

# preserve as a 5d tensor for volumetric architecture



In [ ]:
import os
import torch
import torchio as tio
import ants
import antspynet

def preprocess_3d_mri(input_nifti_path, mni_template_path, target_shape=(1, 128, 128, 128)):
    """
    Complete 3D MRI Preprocessing Pipeline:
    1. Orientation & Bias Field Correction (ANTs)
    2. Deep Learning Skull Stripping (ANTsPyNet)
    3. Affine Registration to MNI152 Space (ANTs)
    4. Resampling & Intensity Normalization (TorchIO)
    """
    # 1. Load Moving MRI and MNI Template
    img = ants.image_read(input_nifti_path)
    template = ants.image_read(mni_template_path)

    # 2. N4 Bias Field Correction
    img_n4 = ants.n4_bias_field_correction(img)

    # 3. ANTsPyNet Deep Learning Skull Stripping
    # Returns a probability mask (0.0 to 1.0)
    prob_mask = antspynet.brain_extraction(img_n4, modality="t1")
    
    # Threshold probability mask at 0.5 to create a crisp binary mask
    binary_mask = ants.threshold_image(prob_mask, low_thresh=0.5, high_thresh=1.0, inval=1, outval=0)
    img_stripped = img_n4 * binary_mask

    # 4. Affine Registration to MNI Template
    reg_to_mni = ants.registration(
        fixed=template, 
        moving=img_stripped, 
        type_of_transform='Affine'
    )
    img_normalized = reg_to_mni['warpedmovout']

    # 5. Direct In-RAM Transfer from ANTs C++ Buffer to TorchIO (No Disk Write)
    tensor_data = torch.from_numpy(img_normalized.numpy()).unsqueeze(0)
    
    # Attach MNI spatial metadata directly
    template_io = tio.ScalarImage(mni_template_path)
    subject = tio.Subject(
        mri=tio.ScalarImage(tensor=tensor_data, affine=template_io.affine)
    )

    # 6. Resampling & Z-Score Intensity Normalization via TorchIO
    transforms = tio.Compose([
        tio.ZNormalization(masking_method=lambda x: x > 0),
        tio.Resize(target_shape[1:])  # Resizes (D, H, W) to (128, 128, 128)
    ])
    
    processed_subject = transforms(subject)

    # 7. Format Output Tensor: (1, 1, D, H, W)
    tensor_4d = processed_subject.mri.data
    tensor_5d = tensor_4d.unsqueeze(0).float() 

    return tensor_5d

In [15]:
preprocess_3d_mri(r"C:\Users\Owner\Downloads\data\nifti_raw\941_S_4376\941_S_4376_MPRAGE_2d.nii.gz", str(MNI152_FILE_PATH))

AttributeError: module 'ants' has no attribute 'image_multiply'

In [ ]:
from pathlib import Path
import torch

load_dotenv()
folder_path_processed = os.getenv('PROCESSED')

patient_directory = Path(folder_path_processed)
output_directory = Path(r'processed_tensors')
output_directory.mkdir(parents=True, exist_ok=True)

for nifti_file in patient_directory.rglob('*.nii.gz'):
    if nifti_file.name.startswith('.'):
        continue

    processed_tensor = preprocess_3d_mri(
        input_nifti_path=str(nifti_file),
        mni_template_path=str(MNI152_FILE_PATH)
    )

    # Save the resulting 5D tensor for model training
    clean_stem = nifti_file.name.replace('.nii.gz', '')
    save_path = output_directory / f"{clean_stem}_preprocessed.pt"
    torch.save(processed_tensor, save_path)

100%|██████████| 12/12 [00:02<00:00,  4.01it/s]


FileNotFoundError: HD-BET ran, but could not locate output mask file in C:\Users\Owner\AppData\Local\Temp\tmp7r6esx8v

In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.5.1+cu121
CUDA Available: True
Device Name: NVIDIA GeForce RTX 3070


In [ ]:
# 1. Path to your top ADNI folder

load_dotenv()
folder_path = os.getenv('ADNI_FOLDER_PATH')
raw_root = Path(folder_path)

# 2. Path to where you want all converted .nii.gz files saved
output_root = Path("data/nifti_raw")

# Loop through every Participant folder inside cn cohort/ADNI
for subject_dir in raw_root.iterdir():
    if subject_dir.is_dir():
        print(f"Converting subject: {subject_dir.name}")
        
        # Create a matching subject directory in your output folder
        subj_output = output_root / subject_dir.name
        subj_output.mkdir(parents=True, exist_ok=True)
        
        # dcm2niix will automatically crawl down into MPRAGE -> Visits -> Cryptic ID -> DICOMs
        dcm2niix_run([
            "-z", "y",                 # Compress output to .nii.gz
            "-f", "%i_%p_%s",          # Filename style: ParticipantID_Protocol_Series
            "-o", str(subj_output),    # Destination folder for this participant
            str(subject_dir)           # Input directory (this participant's root folder)
        ])

print("All participant conversions complete!")